In [0]:
## build the gold customer dimension, one row per customer carrying the surrogate key. No changes from the silver `customers` table structurally since customers were already in a clean, analysis ready shape

from pyspark.sql.functions import col

dim_customer = spark.table("olist.silver.customers").select(
    col("customer_sk"),
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix"),
    col("customer_city"),
    col("customer_state"),
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_customer")

In [0]:
## products dimension
## a denormalised product dimension. Joins the silver `products` table to `category_translation` so the English category name is available directly on the dimension 

from pyspark.sql.functions import col

dim_product = (
    spark.table("olist.silver.products").alias("p")
    .join(spark.table("olist.silver.category_translation").alias("c"),
          col("p.product_category_name") == col("c.product_category_name"), "left")
    .select(
        col("p.product_sk"),
        col("p.product_id"),
        col("p.product_category_name"),
        col("c.product_category_name_english"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm"),
    )
)

dim_product.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_product")

In [0]:
## seller 
## same shape and reasoning as `dim_customer`, one row per seller, with geography attached directly, no separate lookup needed

from pyspark.sql.functions import col

dim_seller = spark.table("olist.silver.sellers").select(
    col("seller_sk"),
    col("seller_id"),
    col("seller_zip_code_prefix"),
    col("seller_city"),
    col("seller_state"),
)

dim_seller.write.format("delta").mode("overwrite").saveAsTable("olist.gold.dim_seller")


In [0]:
## fact_order_items
## Built by joining `order_items` to `orders`, `customers`, `products`, and `sellers` 
## `payments` is aggregated to one row per order first — an order can have multiple payment rows but this table has one row per line item, joining payments directly would multiply each line item once per payment
## Carries the numeric metrics (`price`, `freight_value`, `total_payment_value`) and a date field (`order_date`) needed for the analytics queries in Part A5.

from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import col

payments_agg = (
    spark.table("olist.silver.payments")
    .groupBy("order_id")
    .agg(spark_sum("payment_value").alias("total_payment_value"))
)

fact_order_items = (
    spark.table("olist.silver.order_items").alias("oi")
    .join(spark.table("olist.silver.orders").alias("o"),
        col("oi.order_id") == col("o.order_id"), "left")
    .join(spark.table("olist.silver.customers").alias("c"),
        col("o.customer_id") == col("c.customer_id"), "left")
    .join(spark.table("olist.silver.products").alias("p"),
        col("oi.product_id") == col("p.product_id"), "left")
    .join(spark.table("olist.silver.sellers").alias("s"),
        col("oi.seller_id") == col("s.seller_id"), "left")
    .join(payments_agg.alias("pay"),
        col("oi.order_id") == col("pay.order_id"), "left")
    .select(
        col("oi.order_item_sk"),
        col("o.order_sk"),
        col("c.customer_sk"),
        col("p.product_sk"),
        col("s.seller_sk"),
        col("o.order_purchase_timestamp").alias("order_date"),
        col("o.order_status"),
        col("oi.price"),
        col("oi.freight_value"),
        col("pay.total_payment_value"),
    )
)

fact_order_items.write.format("delta").mode("overwrite").saveAsTable("olist.gold.fact_order_items")



Normalisation happens in the silver layer to keep each entity's data clean, deduplicated and free from redundancy. This makes the conformed layer easy to trust and maintain. We then 
deliberately denormalise into a star schema in gold because analysts can query one wide fact table joined to dimension tables, instead of performing multiple joins across many silver tables.